<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_topical_compling_questions/blob/main/module1_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# КР по модулю 1 — практическая часть



## Задание 1. (1 балл)

Используйте корпус `lenta.txt` из семинара 2 (ячейка ниже скачает его).

In [4]:
!mkdir -p data
!wget https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip -P data
!unzip -o data/lenta.txt.zip -d data/

text = open('data/lenta.txt', encoding='utf-8').read()

--2026-10-29 09:18:35--  https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip [following]
--2026-10-29 09:18:36--  https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5723675 (5.5M) [application/zip]
Saving to: ‘data/lenta.txt.zip’

lenta.txt.zip       100%[===================>]   5.46M  --.-KB/s    in 0.05s   

2026-10-29 09:18:37 (104 MB/s) - ‘data/lenta.txt.zip’ sav

Одним регулярным выражением найдите в **исходном тексте** корпуса (без какой-либо предобработки) все упоминания дат вида «14 сентября»:
- число от 1 до 31 без ведущего нуля (`5`, `14`, `31`, но не `05` и не `32`);
- затем один или несколько пробельных символов;
- затем название месяца в родительном падеже строчными буквами (`января`, …, `декабря`).

Число и месяц должны быть отдельными словами: из `2014 сентября` не должно извлекаться `14 сентября`, а `5 сентябрям` (если бы такое встретилось) не подходит.

In [5]:
import re
dates = re.findall(r"\b(?:[1-9]|[12]\d|3[01])\b\s+(?:января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)\b", text)
print(dates)

['14 сентября', '16 сентября', '16 сентября', '20 сентября', '31 августа', '1 сентября', '1 августа', '17 августа', '30 августа', '30 августа', '30 августа', '24 августа', '24 августа', '31 августа', '7 мая', '18 августа', '26 августа', '1 сентября', '9 августа', '1 сентября', '7 августа', '30 августа', '2 сентября', '17 октября', '1 сентября', '16 сентября', '2 сентября', '30 августа', '28 августа', '22 августа', '19 апреля', '25 августа', '1 мая', '1 сентября', '25 августа', '2 сентября', '30 августа', '28 августа', '13 сентября', '30 августа', '3 сентября', '22 августа', '17 сентября', '4 сентября', '5 сентября', '4 сентября', '31 августа', '5 сентября', '15 декабря', '6 сентября', '30 августа', '7 сентября', '10 сентября', '16 сентября', '23 июля', '29 октября', '27 мая', '6 июля', '6 сентября', '19 сентября', '16 мая', '8 сентября', '5 июля', '14 июля', '22 июля', '6  сентября', '7 сентября', '19 сентября', '7 сентября', '21 сентября', '9 сентября', '9 сентября', '9 сентября', '6 

Посчитайте, сколько раз повторяется каждая дата. Найдите топ-2 даты, вставьте в форму вторую по популярности дату (в таком виде - `25 октября`)

In [6]:
from collections import Counter
counts = Counter(dates)

for date, n in counts.most_common(2):
  print(f"{date}\t{n}")

1 января	131
19 декабря	101


### Задание 2 (1 балл)

Найдите самое вероятное продолжение для биграммы `запрет на` в корпусе `data/lenta.txt`

Для этого приведите весь текст к нижнему регистру и сделайте простую токенизацию - разбейте все по пробелам, удалите пунктуацию по границам (strip) и отфильтруйте пустые строки, оставшиеся после этих шагов. Далее рассчитайте биграммные и триграммные частоты и на их основе вычислите самое вероятное продолжение данной биграммы. Вставьте самое вероятное следующее слово в форму

In [7]:
PUNCT_EDGE = re.compile(r"^[\W_]+|[\W_]+$", flags=re.UNICODE)
tokens = [t2 for t in text.split() if (t2 := PUNCT_EDGE.sub("", t))]

bi  = Counter(zip(tokens, tokens[1:]))
tri = Counter(zip(tokens, tokens[1:], tokens[2:]))

w1, w2 = "запрет", "на"
c_bi = bi[(w1, w2)]

best, best_key = None, None
for (a, b, c), n in tri.items():
  if a == w1 and b == w2:
    key = (n, bi[(w2, c)])
    if best_key is None or key > best_key:
      best, best_key = c, key

print("самое вероятное продолжение:", best)

самое вероятное продолжение: публикацию


### Задание 3 (1 балл)

Загрузите датасет новостей `lenta_40k` из семинара 3 (ячейка ниже). Сделайте подвыборку из 500 текстов: `lenta.sample(500, random_state=42)` (так у всех получится одна и та же выборка)

Посчитайте эмбеддинги этих 500 текстов моделью `intfloat/multilingual-e5-base`, добавив к каждому тексту префикс `passage: `.
Посчитайте эмбеддинг текста `Во Франции сотни тысяч людей снова вышли на протесты, полиция применила слезоточивый газ` (с префиксом `query: `). Найдите самый близкий к этому тексту документ по косинусной близости. Вставьте первое слово этого документа в форму как ответ (регистр не важен).

In [8]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

lenta = pd.read_csv("https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta_40k.csv.zip").sample(500, random_state=42)

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device, "| dtype:", dtype)

model_name = "intfloat/multilingual-e5-base"

model = AutoModel.from_pretrained(
    model_name,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)

@torch.inference_mode()
def embed(texts, prefix, batch_size=32, max_length=512):
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = [f"{prefix}: {t}" for t in texts[i:i + batch_size]]
        enc = tokenizer(
            chunk,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        h = model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        emb = (h * mask).sum(1) / mask.sum(1)
        # нормализацию и приведение к fp32 делаем перед выгрузкой на CPU
        emb = torch.nn.functional.normalize(emb.float(), p=2, dim=1)
        out.append(emb.cpu())
    return torch.cat(out)

lenta = (
    pd.read_csv(
        "https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta_40k.csv.zip",
        compression="zip",
    )
    .sample(500, random_state=42)
    .reset_index(drop=True)
)
print(lenta.shape, lenta.columns.tolist())

doc_emb = embed(lenta["text"].tolist(), "passage")     # (500, 768)
query_emb = embed(
    ["Во Франции сотни тысяч людей снова вышли на протесты, полиция применила слезоточивый газ"],
    "query",
)                                                       # (1, 768)

sims = cosine_similarity(query_emb, doc_emb)[0]
best_doc = lenta.iloc[int(sims.argmax())]["text"]

first_word = re.sub(r"^\W+", "", best_doc.split()[0]).lower()
print("Самый близкий документ:\n", best_doc[:500], "...")

device: cuda | dtype: torch.float16


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(500, 2) ['text', 'topic']
Самый близкий документ:
 Беспорядки в пригородах Парижа, населенных преимущественно иммигрантами из Северной Африки, не стихают уже шестой день подряд. Как сообщает Associated Press со ссылкой на местные власти, столкновения полиции с группами молодежи проходят, по меньшей мере, в девяти городах. Беспорядками охвачен департамент Сена-Сен-Дени, расположенный к северу и северо-востоку от центра Парижа. Количество сожженных автомобилей приближается к 70. Всего в регионе зафиксированы около 150 поджогов. Молодежь забрасывае ...
